In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
drivers = pd.read_csv("../1-Formula_One/drivers.csv")
results = pd.read_csv("../1-Formula_One/results.csv")
driver_standings = pd.read_csv("../1-Formula_One/driver_standings.csv")
races = pd.read_csv("../1-Formula_One/races.csv")
drivers = drivers.drop("number", axis=1)
results = results.drop("number", axis=1)
results = results.merge(races[['year', 'raceId']], on='raceId')
driver_results = pd.merge(drivers, results, on="driverId")
driver_results.to_csv("driver_results.csv", index=False)
driver_results[['dob_year', 'dob_month', 'dob_day']] = driver_results['dob'].str.split('-', expand=True).astype(int)
driver_results = driver_results.drop('dob', axis=1)
#dropping number because it was introduced in 2014 so missing values for older drivers
#cant compare points because there have been many changes surrounding point awarding because the FIA sucks

In [ ]:
wins = driver_results.query('positionOrder == 1').groupby('nationality').agg(
    total_wins = ('positionOrder', 'size')
    ).reset_index()

podiums = driver_results.query('positionOrder <= 3').groupby('nationality').agg(
    podium_placements = ('positionOrder', 'size')
    ).reset_index()

starts = driver_results.groupby('nationality').agg(
    starts = ('positionOrder', 'size')
    ).reset_index()

drivers_data = driver_results.groupby('nationality').agg(
    num_drivers = ('driverId', 'nunique')
    ).reset_index()

avg_pos = driver_results.groupby('nationality').agg(
    average_finish_position = ('positionOrder', 'mean')
).reset_index()

last_race = driver_results.groupby('year')['raceId'].max().reset_index()
champ_info = []
for raceId, year in zip(last_race['raceId'], last_race['year']):
    champion = driver_standings.query('raceId == @raceId and position == 1')[['driverId']]
    champion['year'] = year
    champ_info.append(champion)
champions = pd.concat(champ_info).reset_index(drop=True)
champions= champions.merge(driver_results[['driverId', 'nationality']], on="driverId").groupby('nationality').agg(
    championships = ('year', 'nunique')
).reset_index()

In [ ]:
metrics_q1 = starts.merge(wins, on='nationality', how='left')
metrics_q1 = metrics_q1.merge(podiums,on='nationality', how='left')
metrics_q1 = metrics_q1.merge(drivers_data, on='nationality', how='left')
metrics_q1 = metrics_q1.merge(round(avg_pos), on='nationality', how='left')
metrics_q1 = metrics_q1.merge(champions, on='nationality', how='left')
metrics_q1['win_rate'] = round(((metrics_q1['total_wins'] / metrics_q1['starts']) * 100), 2)
metrics_q1 = metrics_q1.replace(np.nan, 0)
metrics_q1['total_wins'] = metrics_q1['total_wins'].astype(int)
metrics_q1['podium_placements'] = metrics_q1['podium_placements'].astype(int)
metrics_q1['average_finish_position'] = metrics_q1['average_finish_position'].astype(int)
metrics_q1['championships'] = metrics_q1['championships'].astype(int)
display(metrics_q1)

In [ ]:
drivers = pd.read_csv("../1-Formula_One/drivers.csv")
results = pd.read_csv("../1-Formula_One/results.csv")
driver_standings = pd.read_csv("../1-Formula_One/driver_standings.csv")
races = pd.read_csv("../1-Formula_One/races.csv")
status = pd.read_csv("../1-Formula_One/status.csv")
drivers = drivers.drop("number", axis=1)
results = results.drop("number", axis=1)
results = results.merge(races[['year', 'raceId']], on='raceId')
driver_results = pd.merge(drivers, results, on="driverId")
driver_results = driver_results.merge(status, on='statusId', how='left')
driver_results = driver_results.drop('statusId', axis=1)
driver_results['not_dnf'] = driver_results['status'].str.contains("Lap|Finished")
driver_results.to_csv("driver_results.csv", index=False)
driver_results[['dob_year', 'dob_month', 'dob_day']] = driver_results['dob'].str.split('-', expand=True).astype(int)
driver_results = driver_results.drop('dob', axis=1)


In [ ]:
dnf_rate = driver_results.groupby('year').agg(
    dnf_rate=('not_dnf', 'mean')
).reset_index()
dnf_rate['dnf_rate'] = round((1- dnf_rate['dnf_rate']) * 100, 2)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot('year', 'dnf_rate', data=dnf_rate, color='black', linewidth=2)
plt.fill_between(dnf_rate['year'], dnf_rate['dnf_rate'], alpha=0.3, color='black')
plt.xlabel("Year")
plt.ylabel("Did Not Finish (DNF) Percent ")
plt.title("Change in DNF Rate Over Time")
plt.show()

In [ ]:
top_causes = driver_results[driver_results['not_dnf'] == False].groupby('status').agg(
    count = ('status', 'count')
).reset_index().sort_values(by='count', ascending=False)
top_causes

In [ ]:
plt.figure(figsize=(12,6))
plt.bar('status', 'count', data=top_causes.head(10), color='green')
plt.xticks(rotation=45)
plt.xlabel("Cause")
plt.ylabel("Count")
plt.title("Most Common Causes for Not Finishing a Race")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(20, 6))

most_wins = metrics_q1.sort_values('total_wins', ascending=False).query('starts >= 100').head(15)
ax[0].bar('nationality', 'total_wins', data=most_wins, color='purple')
ax[0].set_xticks(range(len(most_wins)))
ax[0].set_xticklabels(most_wins['nationality'], rotation=45)
ax[0].set_xlabel("Nationality")
ax[0].set_ylabel("Total Race Wins")
ax[0].set_title("Comparing Nationality to Race Wins")

best_winrate = metrics_q1.sort_values('win_rate', ascending=False).query('starts >= 100').head(15)
ax[1].bar('nationality', 'win_rate', data=best_winrate, color='darkblue')
ax[1].set_xticks(range(len(best_winrate)))
ax[1].set_xticklabels(best_winrate['nationality'], rotation=45)
ax[1].set_xlabel("Nationality")
ax[1].set_ylabel("Win Rate (Percent)")
ax[1].set_title("Comparing Nationality to Win Rate")

plt.tight_layout()
plt.show()

In [ ]:
metrics_q1 = metrics_q1.sort_values('total_wins', ascending=False)
plt.figure(figsize=(12,6))
plt.bar('nationality', 'total_wins', data=metrics_q1.query('starts >= 100').head(15), color='purple')
plt.xticks(rotation=45)
plt.xlabel("Nationality")
plt.ylabel("Total Race Wins")
plt.title("Comparing Nationality to Race Winnage")
plt.show()

In [ ]:
metrics_q1 = metrics_q1.sort_values('win_rate', ascending=False)
plt.figure(figsize=(12,6))
plt.bar('nationality', 'win_rate', data=metrics_q1.query('starts >= 100').head(15), color='darkblue')
plt.xticks(rotation=45)
plt.xlabel("Nationality")
plt.ylabel("Win Rate (Percent)")
plt.title("Comparing Nationality to Win Rate")
plt.show()

<font size = "10">

**QUESTION 2**

In [ ]:
driver_results['age'] = driver_results['year'] - driver_results['dob_year']
metrics_q2 = driver_results.groupby('year').agg(
    avg_age = ('age', 'mean'),
    unique_nationalities = ('nationality', 'nunique'),
    unique_drivers = ('driverId', 'nunique')
).reset_index()
metrics_q2['avg_age'] = round(metrics_q2['avg_age']).astype(int)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot('year', 'avg_age', data=metrics_q2, color='green', linewidth=2)
plt.fill_between(metrics_q2['year'], metrics_q2['avg_age'], alpha=0.3, color='green')
plt.xlabel("Year")
plt.ylabel("Mean Age")
plt.title("Mean Driver Age Over Time")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot('year','unique_nationalities', data=metrics_q2, color = 'red', linewidth=2)
plt.fill_between(metrics_q2['year'], metrics_q2['unique_nationalities'], alpha=0.3, color='red')
plt.xlabel("Year")
plt.ylabel("Count of Unique Nationalities")
plt.title("Change in Count of Unique Nationalities Over Time")
plt.show()

plt.show()

# **Data Description**

The datasets that we elected to use for our project consisted of four out of the fourteen provided: **drivers.csv**, **results.csv**, 
**driver_standings.csv**, and **races.csv**.

**drivers.csv** contains one observation per driver (854 total) with 9 variables covering driver information.

**results.csv** contains one observation per driver per race (25,660 total) with 18 variables covering outcomes and performance.

**driver_standings.csv** contains one observation per driver per race (33,686 total) tracking *cumulative* championship data for each season.

**races.csv** contains one observation per race (1,079 total) with 18 variables describing race details.

To make working with these datasets easier, we imported `pandas`, `numpy`, and `matplotlib.pyplot`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#change file path as needed
drivers = pd.read_csv("../1-Formula_One/drivers.csv")
results = pd.read_csv("../1-Formula_One/results.csv")
driver_standings = pd.read_csv("../1-Formula_One/driver_standings.csv")
races = pd.read_csv("../1-Formula_One/races.csv")

#to calculate total number of observations and variables
print("Datatable sizes (observations, variables)")
print("drivers.csv size: " + str(drivers.shape))
print("results.csv size: " + str(results.shape))
print("driver_standings.csv size: " + str(driver_standings.shape))
print("races.csv size: " + str(races.shape))

(Prior to any merging the `number` column was dropped from **results.csv** as it was only introduced in 2014, leaving older drivers with missing values and making it unusable for comparisons.)

To condense the relevant information into a single working dataset, we performed two merges. 
First, we merged **races.csv** and **results.csv** on `raceId` to attach the `year` variable to 
each race result. It is important to note, we only took the `year` and `raceId` column from **race.csv**.
Our next merge was to merge **drivers.csv** and the updated **results.csv** on `driverId`, mixing driver information 
with race results. These merges came together to produce our  working dataset, **driver_results**, which contains 
25,660 observations and 28 columns.

In [ ]:
drivers = drivers.drop("number", axis=1)
results = results.drop("number", axis=1)
results = results.merge(races[['year', 'raceId']], on='raceId')
driver_results = pd.merge(drivers, results, on="driverId")

Many steps in cleaning  were taken to prepare the data. The `dob` column was split into three separate integer 
columns — `dob_year`, `dob_month`, and `dob_day` — to allow for age calculations, after which the 
original `dob` column was dropped. It is also important to note that the `points` column in **results.csv** 
was not used during analysis because the FIA has changed the points awarding system multiple times 
throughout F1 which makes using it difficult.

In [ ]:
driver_results[['dob_year', 'dob_month', 'dob_day']] = driver_results['dob'].str.split('-', expand=True).astype(int)
driver_results = driver_results.drop('dob', axis=1)
driver_results.to_csv("driver_results.csv", index=False)

The primary columns used in our analysis were:

`nationality`, `positionOrder`, `driverId`, and `dob_year` from **driver_results**<br/>
`raceId` and `position` from **driver_standings**

Using these columns, we then created two summary datasets. **metrics_q1** was constructed by grouping **driver_results** by 
`nationality` and then aggregating to create: 

`starts` (total race starts)<br/>
`total_wins` (races finished 1st)<br/>
`podium_placements` (races finished top 3)<br/>
`num_drivers` (unique drivers per nationality)<br/>
`average_finish_position` (average `positionOrder`)<br/>
`win_rate` (wins over  starts as a percentage)<br/>

Championship titles per nationality were calculated in a differnt manner. This was done by filtering 
**driver_standings** to the final race of each season and identifying the driver in `position == 1`. After this, we merged that driver back into **metrics_q1**.


**metrics_q2** was constructed by grouping **driver_results** by `year` and aggregating to produce: 

`avg_age` (mean driver age, derived by subtracting `dob_year - year`)<br/>
`unique_nationalities` (distinct nationalities represented on the grid per season)<br/>
`unique_drivers` (distinct drivers per season)<br/>

Any missing values seen after merging in **metrics_q1** were replaced with 0. These were nationalities that had no wins, podiums, or championships.

In [ ]:
display(metrics_q1[['nationality', 'win_rate', 'total_wins']])
display(metrics_q2[['year', 'avg_age','unique_nationalities']])